In [ ]:
! pip install --extra-index-url https://pypi.org/simple openai openai-agents mlflow[kubernetes]

In [ ]:
BASE_URL = "http://llama-32-3b-instruct-predictor.my-first-model.svc.cluster.local:8080/v1"
model_name = "llama-32-3b-instruct"

MCP_SERVER_URL = "http://redhat-openshift-mcp-server.mcp-servers.svc.cluster.local:8080/mcp"

MLFLOW_TRACKING_URL = "https://mlflow.redhat-ods-applications.svc.cluster.local:8443"
MLFLOW_EXPERIMENT = "basic-agent"

input_text = """
List the names of all pods in namespace 'my-first-model'
"""
system_instructions = """
You are a helpful AI assistant.
You are designed to answer questions in a concise and professional manner.
"""

import os
import asyncio
from openai import AsyncOpenAI
from agents import Agent, Runner, set_default_openai_client, set_default_openai_api, set_tracing_disabled
from agents.mcp import MCPServerStreamableHttp
import mlflow

try:
    with open("/var/run/secrets/kubernetes.io/serviceaccount/token", "r") as f:
        AUTH_TOKEN = f.read().strip()
except FileNotFoundError:
    # Fallback if running outside a workbench or using a manually generated cluster token
    AUTH_TOKEN = OCP_TOKEN

os.environ["MLFLOW_TRACKING_AUTH"]="kubernetes-namespaced"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URL)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

mlflow.openai.autolog()

async def main():
    """Main async function to run chat with MCP tools"""

    # Configure custom OpenAI endpoint globally
    custom_client = AsyncOpenAI(
        base_url=BASE_URL,
        api_key=AUTH_TOKEN
    )
    set_default_openai_client(client=custom_client, use_for_tracing=False)
    set_default_openai_api("chat_completions")
    set_tracing_disabled(disabled=True)

    print(f"Connecting to MCP server at {MCP_SERVER_URL}...")

    # Connect to remote MCP server via Streamable HTTP
    async with MCPServerStreamableHttp(
        name="Remote MCP Server",
        params={
            "url": MCP_SERVER_URL,
            "timeout": 30,
        },
        cache_tools_list=True,
    ) as mcp_server:

        # List available tools
        tools = await mcp_server.list_tools()
        print(f"\nFound {len(tools)} tools from MCP server:")

        # Create agent with MCP server
        agent = Agent(
            name="Assistant",
            instructions=system_instructions,
            model=model_name,
            mcp_servers=[mcp_server],
        )

        print(f"\nProcessing query: {input_text.strip()}\n")

        # Run the agent
        result = await Runner.run(agent, input_text)

        # Print the final output
        print("Assistant:", result.final_output)

# Run the async function
if __name__ == "__main__":
    try:
        try:
            get_ipython()
            # Running in Jupyter - use await directly (top-level await is supported)
            import nest_asyncio
            nest_asyncio.apply()
            asyncio.run(main())
        except NameError:
            # Not in Jupyter - use asyncio.run() normally
            asyncio.run(main())
    except Exception as e:
        print(f"An error occurred: {e}")
        import traceback
        traceback.print_exc()
